# OOD probe: PERSUADE 2.0 (student essays, Kaggle Feedback Prize)Runs v4 on 25 essays from the PERSUADE 2.0 test split and computes component-F1 against the schema-mapped gold. See Section 6.2 of the paper.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate

ps -eo pid,cmd | grep pt_main_thread | grep -v grep | awk '{print $1}' | xargs -r kill -9
sleep 2
echo "--- after kill ---"
ps -eo pid,etime,cmd | grep -E "pt_main|python3" | grep -v grep | grep -v ipykernel | grep -v jupyter | grep -v tensorboard | head

# 2. Try to download PERSUADE 2.0 from HuggingFace — probe each candidate
echo ""
echo "=== HF PERSUADE candidates probe ==="
python3 <<'PY'
from datasets import load_dataset
candidates = [
    "realbenpope/PERSUADE_manageable",
    "ruudra1/PERSUADE",
    "introvoyz041/PERSUADE_corpus_2.0",
    "nlpatunt/D_persuade_2",
]
winner = None
for name in candidates:
    try:
        ds = load_dataset(name)
        splits = list(ds.keys())
        first = next(iter(ds[splits[0]]))
        keys = list(first.keys())
        print(f"\n✅ OK: {name}")
        print(f"   splits: {splits}")
        print(f"   n_rows[{splits[0]}]: {len(ds[splits[0]])}")
        print(f"   keys: {keys}")
        # Print first record concisely
        for k in keys[:8]:
            v = first[k]
            preview = str(v)[:200] if v is not None else 'None'
            print(f"   {k}: {preview}")
        if winner is None:
            winner = name
    except Exception as e:
        print(f"❌ FAIL {name}: {str(e)[:140]}")

if winner:
    print(f"\n=== Winner: {winner} ===")
else:
    print("\n=== No HF candidate worked ===")
    print("Falling back to Kaggle-style direct download attempt")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import HfApi
from datasets import load_dataset
import pandas as pd

# ── First, see what files are in the manageable repo ──
api = HfApi()
files = api.list_repo_files("realbenpope/PERSUADE_manageable", repo_type="dataset")
print("=== realbenpope/PERSUADE_manageable files ===")
for f in files: print(f"  {f}")

# ── Try to load a specific file directly via pandas ──
from huggingface_hub import hf_hub_download
try:
    for candidate in ['train.csv', 'PERSUADE_manageable.csv', 'data.csv',
                      'persuade.csv', 'discourse.csv', 'main.csv']:
        try:
            fp = hf_hub_download(repo_id="realbenpope/PERSUADE_manageable",
                                 filename=candidate, repo_type="dataset")
            df = pd.read_csv(fp, nrows=5)
            print(f"\n✅ {candidate} loaded")
            print(f"   columns: {list(df.columns)}")
            print(f"   first row:")
            for c in df.columns[:12]:
                v = str(df.iloc[0][c])[:150]
                print(f"     {c}: {v}")
            break
        except Exception as e:
            pass
except Exception as e:
    print(f"pandas approach failed: {e}")

# ── Also check ruudra1/PERSUADE more carefully ──
print("\n=== ruudra1/PERSUADE files ===")
try:
    files = api.list_repo_files("ruudra1/PERSUADE", repo_type="dataset")
    for f in files: print(f"  {f}")
    # Try loading a specific file
    for candidate in ['train.csv', 'PERSUADE.csv', 'data.csv']:
        try:
            fp = hf_hub_download(repo_id="ruudra1/PERSUADE",
                                 filename=candidate, repo_type="dataset")
            df = pd.read_csv(fp, nrows=5)
            print(f"\n✅ ruudra1/PERSUADE {candidate} loaded")
            print(f"   columns: {list(df.columns)}")
            for c in df.columns[:12]:
                v = str(df.iloc[0][c])[:150]
                print(f"     {c}: {v}")
            break
        except: pass
except Exception as e:
    print(f"ruudra1 probe failed: {e}")

# ── Search HF for feedback-prize alternates ──
print("\n=== HF search: 'feedback prize' ===")
for d in list(api.list_datasets(search="feedback prize", limit=15)):
    print(f"  {d.id}")
print("\n=== HF search: 'discourse element' ===")
for d in list(api.list_datasets(search="discourse element", limit=10)):
    print(f"  {d.id}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import hf_hub_download
import pandas as pd

# ── Try realbenpope/PERSUADE_manageable ──
print("=== realbenpope/PERSUADE_manageable ===")
for fn in ['persuade_corpus_no_full_text.csv', 'persuade_full_text.csv']:
    try:
        fp = hf_hub_download(repo_id="realbenpope/PERSUADE_manageable",
                             filename=fn, repo_type="dataset")
        df = pd.read_csv(fp, nrows=3)
        print(f"\n✅ {fn}  ({len(pd.read_csv(fp, usecols=[0]))} total rows)")
        print(f"   columns: {list(df.columns)}")
        for c in df.columns[:15]:
            v = str(df.iloc[0][c])[:200]
            print(f"     {c}: {v}")
    except Exception as e:
        print(f"\n❌ {fn}: {str(e)[:120]}")

# ── Try ruudra1/PERSUADE ──
print("\n=== ruudra1/PERSUADE ===")
for fn in ['persuade_corpus_2.0_train.csv', 'persuade_corpus_2.0_test.csv']:
    try:
        fp = hf_hub_download(repo_id="ruudra1/PERSUADE",
                             filename=fn, repo_type="dataset")
        df = pd.read_csv(fp, nrows=3)
        print(f"\n✅ {fn}  ({len(pd.read_csv(fp, usecols=[0]))} total rows)")
        print(f"   columns: {list(df.columns)}")
        for c in df.columns[:15]:
            v = str(df.iloc[0][c])[:200]
            print(f"     {c}: {v}")
    except Exception as e:
        print(f"\n❌ {fn}: {str(e)[:120]}")

# ── Now that we know schemas, look at label distribution in the winner ──
print("\n=== label distribution (whichever loaded) ===")
try:
    fp = hf_hub_download(repo_id="realbenpope/PERSUADE_manageable",
                         filename='persuade_corpus_no_full_text.csv', repo_type="dataset")
    df = pd.read_csv(fp)
    for col_candidate in ['discourse_type', 'label', 'type', 'category']:
        if col_candidate in df.columns:
            print(f"{col_candidate} distribution:")
            print(df[col_candidate].value_counts())
            break
except Exception as e:
    print(f"failed: {e}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p eval_logs phase2_data/raw/persuade
nohup python3 -u <<'PY' > eval_logs/persuade_v4.log 2>&1 &
import sys, json, random, re, time
sys.path.insert(0, '.')
import pandas as pd
from huggingface_hub import hf_hub_download
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

# ── Load model ──
cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 2048   # essays are longer than AMPERSAND sentences
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print("loaded v4 (patched _parse_output)\n", flush=True)

# ── Load PERSUADE 2.0 test split ──
fp = hf_hub_download(repo_id="ruudra1/PERSUADE",
                     filename='persuade_corpus_2.0_test.csv', repo_type="dataset")
df = pd.read_csv(fp)
print(f"PERSUADE test rows: {len(df)}, essays: {df['essay_id_comp'].nunique()}\n", flush=True)

# ── Role mapping ──
CLAIM_LIKE   = {'Claim', 'Position', 'Counterclaim', 'Concluding Statement'}
PREMISE_LIKE = {'Evidence', 'Rebuttal'}
EXCLUDED     = {'Lead', 'Unannotated'}   # non-argumentative filler or introductory

def map_role(t):
    if t in CLAIM_LIKE:   return 'claim'
    if t in PREMISE_LIKE: return 'premise'
    return None  # excluded

# ── Group by essay ──
essays = []
for eid, grp in df.groupby('essay_id_comp'):
    ft = str(grp['full_text'].iloc[0])
    if len(ft) < 100 or ft == 'nan': continue
    gold = []
    for _, r in grp.iterrows():
        role = map_role(str(r['discourse_type']))
        if role and pd.notna(r['discourse_text']):
            gold.append({'text': str(r['discourse_text']), 'role': role})
    if gold: essays.append({'id': eid, 'text': ft, 'gold': gold})
print(f"essays with usable gold: {len(essays)}\n", flush=True)

random.seed(42)
sample = random.sample(essays, min(25, len(essays)))
print(f"sample: {len(sample)} essays\n", flush=True)

# ── Word-Jaccard span match ──
def toks(s): return set(re.findall(r'\w+', s.lower()))
def overlap(a, b):
    ta, tb = toks(a), toks(b)
    return (len(ta & tb) / len(ta | tb)) if (ta and tb) else 0.0

tp_c=fp_c=fn_c=tp_p=fp_p=fn_p=0
n_ext=n_gtot=n_real=0
t_start = time.time()

for idx, e in enumerate(sample, 1):
    gold = e['gold']
    text = e['text'][:8000]  # keep tokenizer input tractable
    t = time.time()
    try:
        pred, _ = student.predict(text)
    except Exception as ex:
        print(f"[{idx}/{len(sample)}] {e['id']}: ERR {ex}", flush=True); continue
    dt = time.time() - t
    pred_spans = ([{'text':c.get('text',''),'role':'claim'}   for c in pred['claim_components']   if c.get('text')] +
                  [{'text':p.get('text',''),'role':'premise'} for p in pred['premise_components'] if p.get('text')])
    gc = sum(1 for g in gold if g['role']=='claim'); gp = len(gold)-gc

    if not pred_spans:
        print(f"[{idx}/{len(sample)}] {e['id']}: EMPTY t={dt:.0f}s gold_c/p={gc}/{gp}", flush=True)
        fn_c += gc; fn_p += gp; n_gtot += len(gold); continue

    mg = [False]*len(gold); mp = [False]*len(pred_spans)
    for pi, ps in enumerate(pred_spans):
        best_gi, best_ov = -1, 0.0
        for gi, gs in enumerate(gold):
            if mg[gi]: continue
            ov = overlap(ps['text'], gs['text'])
            if ov > best_ov: best_gi, best_ov = gi, ov
        if best_ov >= 0.5:
            mg[best_gi]=True; mp[pi]=True
            if ps['role']==gold[best_gi]['role']:
                if ps['role']=='claim': tp_c+=1
                else: tp_p+=1
            else:
                if ps['role']=='claim': fp_c+=1
                else: fp_p+=1
                if gold[best_gi]['role']=='claim': fn_c+=1
                else: fn_p+=1
    for pi,m in enumerate(mp):
        if not m:
            if pred_spans[pi]['role']=='claim': fp_c+=1
            else: fp_p+=1
    for gi,m in enumerate(mg):
        if not m:
            if gold[gi]['role']=='claim': fn_c+=1
            else: fn_p+=1

    n_ext += sum(mp); n_gtot += len(gold); n_real += 1
    pc = len(pred['claim_components']); pp = len(pred['premise_components'])
    print(f"[{idx}/{len(sample)}] {e['id']}: pred_c/p={pc}/{pp} gold_c/p={gc}/{gp} t={dt:.0f}s", flush=True)

def f1(tp,fp,fn):
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r

f1c,pc,rc = f1(tp_c,fp_c,fn_c); f1p,pp,rp = f1(tp_p,fp_p,fn_p)
print(f"\n=== PERSUADE 2.0 (non-empty {n_real}/{len(sample)}) ===")
print(f"claim   F1={f1c:.3f}  P={pc:.3f}  R={rc:.3f}  (tp={tp_c} fp={fp_c} fn={fn_c})")
print(f"premise F1={f1p:.3f}  P={pp:.3f}  R={rp:.3f}  (tp={tp_p} fp={fp_p} fn={fn_p})")
print(f"COMPONENT F1 (macro): {(f1c+f1p)/2:.3f}")
print(f"extraction rate: {n_ext}/{n_gtot} = {n_ext/max(n_gtot,1):.1%}")
print(f"wall: {(time.time()-t_start)/60:.1f} min")
PY
echo "PID: $!"
sleep 3
tail -10 eval_logs/persuade_v4.log

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate

pip install -q --upgrade huggingface_hub

# Also confirm where things ended up
which huggingface-cli || echo "CLI not on PATH — no problem, we'll use Python"
python3 -c "import huggingface_hub; print('hub version:', huggingface_hub.__version__)"

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
from huggingface_hub import login
# Paste your WRITE token here — starts with hf_
login(token="", add_to_git_credential=False)

# Verify
from huggingface_hub import whoami
me = whoami()
print(f"logged in as: {me['name']} (email: {me.get('email','?')})")
PY